In [ ]:
# ================================
# 🚀 FULL PIPELINE: SSDD SHIP DETECTION
# ================================

# ----------------
# 1. Imports
# ----------------
import os
import cv2
import numpy as np
import xml.etree.ElementTree as ET
import tensorflow as tf
import matplotlib.pyplot as plt

from glob import glob
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam

# ----------------
# 2. Mount Drive
# ----------------
from google.colab import drive
drive.mount('/content/drive')

# ----------------
# 3. Paths
# ----------------
SSDD_PATH = "/content/drive/MyDrive/Official-SSDD-OPEN/BBox_SSDD/voc_style"

TRAIN_IMG_PATH = os.path.join(SSDD_PATH, "JPEGImages_train")
TRAIN_MASK_PATH = os.path.join(SSDD_PATH, "Masks_train")

TEST_IMG_PATH = os.path.join(SSDD_PATH, "JPEGImages_test")
TEST_MASK_PATH = os.path.join(SSDD_PATH, "Masks_test")
TEST_ANN_PATH = os.path.join(SSDD_PATH, "Annotations_test")

IMG_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 10

# ----------------
# 4. Load Data
# ----------------
def load_image_mask(img_path, mask_path):
    img = cv2.imread(img_path)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0

    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE))
    mask = np.expand_dims(mask, axis=-1) / 255.0

    return img, mask

def load_dataset(img_dir, mask_dir):
    images = sorted(glob(os.path.join(img_dir, "*.jpg")))
    masks = sorted(glob(os.path.join(mask_dir, "*.png")))

    X, Y = [], []
    for img_path, mask_path in zip(images, masks):
        img, mask = load_image_mask(img_path, mask_path)
        X.append(img)
        Y.append(mask)

    return np.array(X), np.array(Y), images, masks

print("📦 Loading Dataset...")
X_train, Y_train, _, _ = load_dataset(TRAIN_IMG_PATH, TRAIN_MASK_PATH)
X_test, Y_test, test_images, test_masks = load_dataset(TEST_IMG_PATH, TEST_MASK_PATH)

print("✅ Dataset Loaded")

# ----------------
# 5. U-Net Model
# ----------------
def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    return x

def build_unet():
    inputs = layers.Input((IMG_SIZE, IMG_SIZE, 3))

    c1 = conv_block(inputs, 32)
    p1 = layers.MaxPooling2D()(c1)

    c2 = conv_block(p1, 64)
    p2 = layers.MaxPooling2D()(c2)

    c3 = conv_block(p2, 128)
    p3 = layers.MaxPooling2D()(c3)

    c4 = conv_block(p3, 256)
    p4 = layers.MaxPooling2D()(c4)

    c5 = conv_block(p4, 512)

    u6 = layers.UpSampling2D()(c5)
    u6 = layers.Concatenate()([u6, c4])
    c6 = conv_block(u6, 256)

    u7 = layers.UpSampling2D()(c6)
    u7 = layers.Concatenate()([u7, c3])
    c7 = conv_block(u7, 128)

    u8 = layers.UpSampling2D()(c7)
    u8 = layers.Concatenate()([u8, c2])
    c8 = conv_block(u8, 64)

    u9 = layers.UpSampling2D()(c8)
    u9 = layers.Concatenate()([u9, c1])
    c9 = conv_block(u9, 32)

    outputs = layers.Conv2D(1, 1, activation="sigmoid")(c9)

    return models.Model(inputs, outputs)

model = build_unet()
model.compile(optimizer=Adam(1e-4), loss="binary_crossentropy", metrics=["accuracy"])

model.summary()

# ----------------
# 6. Train Model
# ----------------
print("🚀 Training...")
history = model.fit(
    X_train, Y_train,
    validation_data=(X_test, Y_test),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE
)

# ----------------
# 7. IoU Function
# ----------------
def compute_iou(box1, box2):
    xA = max(box1[0], box2[0])
    yA = max(box1[1], box2[1])
    xB = min(box1[2], box2[2])
    yB = min(box1[3], box2[3])

    inter = max(0, xB - xA) * max(0, yB - yA)

    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])

    union = area1 + area2 - inter

    return inter / union if union > 0 else 0

# ----------------
# 8. Extract BBoxes
# ----------------
def get_gt_boxes(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    boxes = []
    for obj in root.findall("object"):
        bbox = obj.find("bndbox")
        xmin = int(bbox.find("xmin").text)
        ymin = int(bbox.find("ymin").text)
        xmax = int(bbox.find("xmax").text)
        ymax = int(bbox.find("ymax").text)
        boxes.append([xmin, ymin, xmax, ymax])

    return boxes

def get_pred_boxes(mask):
    mask = (mask > 0.5).astype(np.uint8) * 255
    mask = mask.squeeze()

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    boxes = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        boxes.append([x, y, x+w, y+h])

    return boxes

# ----------------
# 9. Compute mAP
# ----------------
def compute_map(test_images, test_masks, model):
    aps = []

    for i in range(len(test_images)):

        # Load GT
        filename = os.path.basename(test_images[i]).split(".")[0]
        xml_path = os.path.join(TEST_ANN_PATH, filename + ".xml")

        if not os.path.exists(xml_path):
            continue

        gt_boxes = get_gt_boxes(xml_path)

        # Predict
        img = cv2.imread(test_images[i])
        img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE)) / 255.0

        pred_mask = model.predict(np.expand_dims(img_resized, axis=0))[0]

        pred_boxes = get_pred_boxes(pred_mask)

        # Compute AP
        tp, fp = 0, 0

        for pb in pred_boxes:
            best_iou = 0
            for gb in gt_boxes:
                best_iou = max(best_iou, compute_iou(pb, gb))

            if best_iou > 0.5:
                tp += 1
            else:
                fp += 1

        ap = tp / (tp + fp) if (tp + fp) > 0 else 0
        aps.append(ap)

        print(f"Image {i+1}: AP = {ap:.4f}")

    return np.mean(aps)

# ----------------
# 10. Final mAP
# ----------------
print("\n📊 Evaluating mAP...")
map_score = compute_map(test_images, test_masks, model)

print("\n🚀 FINAL RESULT")
print(f"✅ mAP: {map_score:.4f}")

# ----------------
# 11. Visualization
# ----------------
def show_sample(idx):
    img = cv2.imread(test_images[idx])
    img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE)) / 255.0

    pred_mask = model.predict(np.expand_dims(img_resized, axis=0))[0]

    plt.figure(figsize=(10,4))

    plt.subplot(1,2,1)
    plt.imshow(img)
    plt.title("Original")

    plt.subplot(1,2,2)
    plt.imshow(pred_mask.squeeze(), cmap='gray')
    plt.title("Prediction")

    plt.show()

show_sample(0)